# LendingClub Loan Risk Analysis

This notebook inspects the accepted-loans dataset before defining the modeling population.

In [6]:
from pathlib import Path
import duckdb
import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

csv_path = project_dir / "data" / "accepted_2007_to_2018Q4.csv"
db_path = project_dir / "lendingclub.duckdb"

assert csv_path.exists(), f"Dataset not found: {csv_path}"
con = duckdb.connect(str(db_path))

print(f"DuckDB version: {duckdb.__version__}")
print(f"Dataset: {csv_path}")


DuckDB version: 1.4.0
Dataset: /Users/meetshah/Desktop/lendingclub-risk-analysis/data/accepted_2007_to_2018Q4.csv


## Inspect loan outcomes

We count every status before filtering so the target definition is explicit and reproducible.

In [7]:
status_counts = con.execute("""
    SELECT
        loan_status,
        COUNT(*) AS loan_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM read_csv_auto(?, ignore_errors=true)
    GROUP BY loan_status
    ORDER BY loan_count DESC
""", [str(csv_path)]).df()

status_counts

,loan_status,loan_count,percentage
0,Fully Paid,1076751,47.63
1,Current,878317,38.85
2,Charged Off,268559,11.88
3,Late (31-120 days),21467,0.95
4,In Grace Period,8436,0.37
5,Late (16-30 days),4349,0.19
6,Does not meet the credit policy. Status:Fully ...,1988,0.09
7,Does not meet the credit policy. Status:Charge...,761,0.03
8,Default,40,0.00
9,None,33,0.00


In [8]:
con.execute("""
    CREATE OR REPLACE TABLE loans_resolved AS
    SELECT
        id,
        loan_amnt,
        funded_amnt,
        term,
        int_rate,
        installment,
        grade,
        sub_grade,
        emp_length,
        home_ownership,
        annual_inc,
        verification_status,
        issue_d,
        purpose,
        addr_state,
        dti,
        delinq_2yrs,
        earliest_cr_line,
        fico_range_low,
        fico_range_high,
        inq_last_6mths,
        open_acc,
        pub_rec,
        revol_bal,
        revol_util,
        total_acc,
        application_type,
        mort_acc,
        pub_rec_bankruptcies,
        loan_status,
        CASE
            WHEN loan_status IN ('Charged Off', 'Default') THEN 1
            WHEN loan_status = 'Fully Paid' THEN 0
        END AS is_default
    FROM read_csv_auto(?, ignore_errors=true)
    WHERE loan_status IN ('Fully Paid', 'Charged Off', 'Default')
""", [str(csv_path)])

print("Filtered table created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtered table created.


In [9]:
con.execute("""
    SELECT
        loan_status,
        is_default,
        COUNT(*) AS loan_count
    FROM loans_resolved
    GROUP BY loan_status, is_default
    ORDER BY is_default, loan_status
""").df()

,loan_status,is_default,loan_count
0,Fully Paid,0,1076751
1,Charged Off,1,268559
2,Default,1,40


## Portfolio Overview

This section summarizes the resolved-loan population and its observed charge-off risk.

In [10]:
portfolio_overview = con.execute("""
    SELECT
        COUNT(*) AS total_loans,
        ROUND(SUM(loan_amnt), 0) AS originated_principal,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate
    FROM loans_resolved
""").df()

portfolio_overview

,total_loans,originated_principal,average_loan_amount,average_interest_rate,charge_off_rate
0,1345350,1.939991e+10,14419.97,13.24,19.96


## Credit Risk by Loan Grade

Loan grade is LendingClub's assigned credit-risk category, ranging from A (lowest assessed risk) to G (highest assessed risk).

In [11]:
risk_by_grade = con.execute("""
    SELECT
        grade,
        COUNT(*) AS loan_count,
        ROUND(SUM(loan_amnt), 0) AS originated_principal,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)
            AS share_of_loans,
        ROUND(100.0 * SUM(loan_amnt) / SUM(SUM(loan_amnt)) OVER (), 2)
            AS share_of_principal,
        ROUND(100.0 * AVG(is_default), 2)
            AS charge_off_rate,
        ROUND(AVG(int_rate), 2)
            AS average_interest_rate
    FROM loans_resolved
    GROUP BY grade
    ORDER BY grade
""").df()

risk_by_grade

,grade,loan_count,originated_principal,share_of_loans,share_of_principal,charge_off_rate,average_interest_rate
0,A,235095,3.266021e+09,17.47,16.84,6.04,7.11
1,B,392748,5.199050e+09,29.19,26.80,13.39,10.68
2,C,381694,5.415625e+09,28.37,27.92,22.44,14.02
3,D,200966,3.069207e+09,14.94,15.82,30.39,17.72
4,E,93656,1.650055e+09,6.96,8.51,38.48,21.14
5,F,32059,6.119320e+08,2.38,3.15,45.20,24.93
6,G,9132,1.880170e+08,0.68,0.97,49.93,27.73


## Charged-Off Loan Exposure by Grade

This analysis measures the original principal associated with loans that were eventually charged off or defaulted. It does not represent actual financial loss because borrowers may have made partial payments before charge-off.

In [12]:
charged_off_exposure = con.execute("""
    SELECT
        grade,
        COUNT(*) AS total_loans,
        SUM(is_default) AS charged_off_loans,

        ROUND(
            100.0 * AVG(is_default),
            2
        ) AS charge_off_rate,

        ROUND(
            SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END),
            0
        ) AS charged_off_loan_principal,

        ROUND(
            100.0
            * SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END)
            / SUM(
                SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END)
              ) OVER (),
            2
        ) AS share_of_charged_off_principal

    FROM loans_resolved
    GROUP BY grade
    ORDER BY charged_off_loan_principal DESC
""").df()

charged_off_exposure

,grade,total_loans,charged_off_loans,charge_off_rate,charged_off_loan_principal,share_of_charged_off_principal
0,C,381694,85657.0,22.44,1.263444e+09,30.22
1,D,200966,61067.0,30.39,9.776938e+08,23.39
2,B,392748,52576.0,13.39,7.128407e+08,17.05
3,E,93656,36041.0,38.48,6.542715e+08,15.65
4,F,32059,14492.0,45.20,2.838272e+08,6.79
5,A,235095,14206.0,6.04,1.951735e+08,4.67
6,G,9132,4560.0,49.93,9.345935e+07,2.24


In [13]:
risk_by_term = con.execute("""
    SELECT
        TRIM(term) AS loan_term,
        COUNT(*) AS loan_count,
        ROUND(SUM(loan_amnt), 0) AS originated_principal,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate,
        ROUND(
            SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END),
            0
        ) AS charged_off_loan_principal
    FROM loans_resolved
    GROUP BY TRIM(term)
    ORDER BY loan_term
""").df()

risk_by_term

,loan_term,loan_count,originated_principal,average_loan_amount,average_interest_rate,charge_off_rate,charged_off_loan_principal
0,36 months,1020768,1.280822e+10,12547.63,12.12,16.00,2.057536e+09
1,60 months,324582,6.591688e+09,20308.24,16.77,32.45,2.123173e+09


In [14]:
term_by_grade = con.execute("""
    SELECT
        grade,
        TRIM(term) AS loan_term,
        COUNT(*) AS loan_count,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate
    FROM loans_resolved
    GROUP BY grade, TRIM(term)
    ORDER BY grade, loan_term
""").df()

term_by_grade

,grade,loan_term,loan_count,average_loan_amount,average_interest_rate,charge_off_rate
0,A,36 months,228808,13743.68,7.09,5.92
1,A,60 months,6287,19302.72,7.87,10.61
2,B,36 months,344939,12244.89,10.68,12.75
3,B,60 months,47809,20400.15,10.70,18.01
4,C,36 months,276119,12038.16,13.93,20.49
5,C,60 months,105575,19812.07,14.27,27.53
6,D,36 months,126065,12401.00,17.68,26.89
7,D,60 months,74901,20104.87,17.79,36.27
8,E,36 months,36035,12439.25,21.23,32.70
9,E,60 months,57621,20857.09,21.08,42.10


In [15]:
risk_by_purpose = con.execute("""
    SELECT
        purpose,
        COUNT(*) AS loan_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_of_loans,
        ROUND(SUM(loan_amnt), 0) AS originated_principal,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate,
        ROUND(
            SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END),
            0
        ) AS charged_off_loan_principal
    FROM loans_resolved
    GROUP BY purpose
    HAVING COUNT(*) >= 1000
    ORDER BY charge_off_rate DESC
""").df()

risk_by_purpose

,purpose,loan_count,share_of_loans,originated_principal,average_loan_amount,average_interest_rate,charge_off_rate,charged_off_loan_principal
0,small_business,15416,1.15,2.411311e+08,15641.61,15.94,29.71,7.815975e+07
1,moving,9480,0.71,7.454552e+07,7863.45,15.15,23.35,1.988375e+07
2,house,7254,0.54,1.117219e+08,15401.42,15.42,21.89,2.564142e+07
3,medical,15556,1.16,1.400346e+08,9001.96,14.01,21.79,3.472378e+07
4,debt_consolidation,780342,58.06,1.188647e+10,15232.38,13.62,21.15,2.677554e+09
5,other,77877,5.79,7.654911e+08,9829.49,14.59,21.04,1.874929e+08
6,vacation,9065,0.67,5.613358e+07,6192.34,13.70,19.17,1.425310e+07
7,major_purchase,29427,2.19,3.483717e+08,11838.50,12.74,18.61,8.237170e+07
8,home_improvement,87507,6.51,1.237774e+09,14144.86,12.79,17.72,2.480818e+08
9,credit_card,295285,21.97,4.373462e+09,14810.99,11.79,16.93,7.840514e+08


In [16]:
risk_by_dti = con.execute("""
    SELECT
        CASE
            WHEN dti IS NULL THEN 'Missing'
            WHEN dti < 0 THEN 'Invalid'
            WHEN dti < 10 THEN '00–09.99'
            WHEN dti < 20 THEN '10–19.99'
            WHEN dti < 30 THEN '20–29.99'
            WHEN dti < 40 THEN '30–39.99'
            ELSE '40+'
        END AS dti_band,

        COUNT(*) AS loan_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_of_loans,

        ROUND(AVG(dti), 2) AS average_dti,

        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,

        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate

    FROM loans_resolved
    GROUP BY dti_band
    ORDER BY
        CASE dti_band
            WHEN '00–09.99' THEN 1
            WHEN '10–19.99' THEN 2
            WHEN '20–29.99' THEN 3
            WHEN '30–39.99' THEN 4
            WHEN '40+' THEN 5
            WHEN 'Missing' THEN 6
            ELSE 7
        END
""").df()

risk_by_dti

,dti_band,loan_count,share_of_loans,average_dti,average_loan_amount,charge_off_rate
0,00–09.99,245721,18.26,6.47,13548.97,14.89
1,10–19.99,561921,41.77,15.10,14620.65,17.84
2,20–29.99,408647,30.37,24.41,14711.04,23.04
3,30–39.99,121915,9.06,33.44,14100.54,29.09
4,40+,6770,0.50,68.46,17401.48,30.55
5,Missing,374,0.03,NaN,17270.92,18.98
6,Invalid,2,0.00,-1.00,16000.00,0.00


In [17]:
risk_by_fico = con.execute("""
    WITH fico_data AS (
        SELECT
            *,
            (fico_range_low + fico_range_high) / 2.0 AS fico_score
        FROM loans_resolved
    )
    SELECT
        CASE
            WHEN fico_score IS NULL THEN 'Missing'
            WHEN fico_score < 650 THEN 'Below 650'
            WHEN fico_score < 700 THEN '650–699'
            WHEN fico_score < 750 THEN '700–749'
            WHEN fico_score < 800 THEN '750–799'
            ELSE '800+'
        END AS fico_band,

        COUNT(*) AS loan_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_of_loans,

        ROUND(AVG(fico_score), 1) AS average_fico,

        ROUND(AVG(int_rate), 2) AS average_interest_rate,

        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate

    FROM fico_data
    GROUP BY fico_band
    ORDER BY
        CASE fico_band
            WHEN 'Below 650' THEN 1
            WHEN '650–699' THEN 2
            WHEN '700–749' THEN 3
            WHEN '750–799' THEN 4
            WHEN '800+' THEN 5
            ELSE 6
        END
""").df()

risk_by_fico

,fico_band,loan_count,share_of_loans,average_fico,average_interest_rate,charge_off_rate
0,Below 650,2,0.00,629.5,15.33,0.00
1,650–699,820825,61.01,677.9,14.56,23.59
2,700–749,417917,31.06,718.3,11.76,15.67
3,750–799,91158,6.78,769.0,9.06,9.31
4,800+,15448,1.15,813.5,7.90,6.42


In [18]:
maturity_audit = con.execute("""
    WITH all_loans AS (
        SELECT
            YEAR(STRPTIME(issue_d, '%b-%Y')) AS issue_year,
            loan_status
        FROM read_csv_auto(?, ignore_errors=true)
        WHERE issue_d IS NOT NULL
    )
    SELECT
        issue_year,
        COUNT(*) AS total_loans,
        SUM(
            CASE
                WHEN loan_status IN ('Fully Paid', 'Charged Off', 'Default')
                THEN 1 ELSE 0
            END
        ) AS resolved_loans,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN loan_status IN ('Fully Paid', 'Charged Off', 'Default')
                    THEN 1 ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS resolved_rate
    FROM all_loans
    GROUP BY issue_year
    ORDER BY issue_year
""", [str(csv_path)]).df()

maturity_audit

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,issue_year,total_loans,resolved_loans,resolved_rate
0,2007,603,251.0,41.63
1,2008,2393,1562.0,65.27
2,2009,5281,4716.0,89.30
3,2010,12537,11536.0,92.02
4,2011,21721,21721.0,100.00
5,2012,53367,53367.0,100.00
6,2013,134814,134804.0,99.99
7,2014,235629,223103.0,94.68
8,2015,421095,375546.0,89.18
9,2016,434407,293105.0,67.47


In [19]:
con.execute("""
    CREATE OR REPLACE TABLE loans_mature AS
    SELECT *
    FROM loans_resolved
    WHERE YEAR(STRPTIME(issue_d, '%b-%Y'))
          BETWEEN 2011 AND 2013
""")

con.execute("""
    SELECT
        COUNT(*) AS mature_loans,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate,
        ROUND(SUM(loan_amnt), 0) AS originated_principal
    FROM loans_mature
""").df()

,mature_loans,charge_off_rate,originated_principal
0,209892,15.71,2.962708e+09


In [20]:
mature_term = con.execute("""
    SELECT
        TRIM(term) AS loan_term,
        COUNT(*) AS loan_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)
            AS share_of_loans,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate,
        ROUND(
            SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END),
            0
        ) AS charged_off_loan_principal
    FROM loans_mature
    GROUP BY TRIM(term)
    ORDER BY loan_term
""").df()

mature_grade = con.execute("""
    SELECT
        grade,
        COUNT(*) AS loan_count,
        ROUND(SUM(loan_amnt), 0) AS originated_principal,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate,
        ROUND(AVG(int_rate), 2) AS average_interest_rate,
        ROUND(
            100.0
            * SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END)
            / SUM(
                SUM(CASE WHEN is_default = 1 THEN loan_amnt ELSE 0 END)
              ) OVER (),
            2
        ) AS share_of_charged_off_principal
    FROM loans_mature
    GROUP BY grade
    ORDER BY grade
""").df()

mature_term_by_grade = con.execute("""
    SELECT
        grade,
        TRIM(term) AS loan_term,
        COUNT(*) AS loan_count,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate
    FROM loans_mature
    GROUP BY grade, TRIM(term)
    ORDER BY grade, loan_term
""").df()

display(mature_term)
display(mature_grade)
display(mature_term_by_grade)

,loan_term,loan_count,share_of_loans,average_loan_amount,average_interest_rate,charge_off_rate,charged_off_loan_principal
0,36 months,157993,75.27,12106.47,12.91,12.52,231209900.0
1,60 months,51899,24.73,20231.02,17.58,25.41,269263250.0


,grade,loan_count,originated_principal,charge_off_rate,average_interest_rate,share_of_charged_off_principal
0,A,34334,443918575.0,5.80,7.60,4.80
1,B,69187,890205775.0,11.10,11.83,19.66
2,C,53947,768562175.0,17.77,15.36,27.70
3,D,30685,434759225.0,22.70,18.41,20.65
4,E,13983,262677175.0,28.83,21.16,15.60
5,F,6429,131842975.0,34.58,23.39,9.35
6,G,1327,30741950.0,36.62,24.77,2.24


,grade,loan_term,loan_count,average_loan_amount,charge_off_rate
0,A,36 months,33389,12829.36,5.73
1,A,60 months,945,16464.52,8.36
2,B,36 months,61840,12116.67,10.60
3,B,60 months,7347,19179.42,15.30
4,C,36 months,36798,11870.44,15.92
5,C,60 months,17149,19345.37,21.75
6,D,36 months,20854,11356.39,20.44
7,D,60 months,9831,20133.56,27.48
8,E,36 months,4298,12079.56,22.87
9,E,60 months,9685,21761.41,31.47


In [21]:
mature_dti = con.execute("""
    SELECT
        CASE
            WHEN dti IS NULL OR dti < 0 THEN 'Missing/Invalid'
            WHEN dti < 10 THEN '00–09.99'
            WHEN dti < 20 THEN '10–19.99'
            WHEN dti < 30 THEN '20–29.99'
            WHEN dti < 40 THEN '30–39.99'
            ELSE '40+'
        END AS dti_band,
        COUNT(*) AS loan_count,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate
    FROM loans_mature
    GROUP BY dti_band
    ORDER BY dti_band
""").df()

mature_fico = con.execute("""
    WITH fico_data AS (
        SELECT
            *,
            (fico_range_low + fico_range_high) / 2.0 AS fico_score
        FROM loans_mature
    )
    SELECT
        CASE
            WHEN fico_score < 650 THEN 'Below 650'
            WHEN fico_score < 700 THEN '650–699'
            WHEN fico_score < 750 THEN '700–749'
            WHEN fico_score < 800 THEN '750–799'
            ELSE '800+'
        END AS fico_band,
        COUNT(*) AS loan_count,
        ROUND(100.0 * AVG(is_default), 2) AS charge_off_rate
    FROM fico_data
    GROUP BY fico_band
    ORDER BY MIN(fico_score)
""").df()

missingness_audit = con.execute("""
    SELECT 'annual_inc' AS feature, COUNT(*) - COUNT(annual_inc) AS missing
    FROM loans_mature
    UNION ALL
    SELECT 'dti', COUNT(*) - COUNT(dti) FROM loans_mature
    UNION ALL
    SELECT 'emp_length', COUNT(*) - COUNT(emp_length) FROM loans_mature
    UNION ALL
    SELECT 'revol_util', COUNT(*) - COUNT(revol_util) FROM loans_mature
    UNION ALL
    SELECT 'mort_acc', COUNT(*) - COUNT(mort_acc) FROM loans_mature
    UNION ALL
    SELECT 'pub_rec_bankruptcies',
           COUNT(*) - COUNT(pub_rec_bankruptcies)
    FROM loans_mature
""").df()

display(mature_dti)
display(mature_fico)
display(missingness_audit)

,dti_band,loan_count,charge_off_rate
0,00–09.99,43651,11.98
1,10–19.99,95092,15.00
2,20–29.99,61622,18.68
3,30–39.99,9527,20.53


,fico_band,loan_count,charge_off_rate
0,650–699,120351,18.71
1,700–749,72291,12.76
2,750–799,15080,7.54
3,800+,2170,4.29


,feature,missing
0,annual_inc,0
1,dti,0
2,emp_length,8646
3,revol_util,134
4,mort_acc,29216
5,pub_rec_bankruptcies,0
